# M49 Kaggle — frozen dev-200 evaluation

Chỉ chạy sau notebook training đã có manifest `complete: true`. Add các input: `legal-agentic-rag-m49-source.zip`; official `train.json`; `Scoring-Program-Task-LegalQA.zip`; output M45 có `uit-dsc-2026-task2-m45-artifacts.tar.gz` và `.sha256`; output notebook M49 training có thư mục `m49-qwen3.5-2b-official-sft-v1`. Bật Internet và GPU T4 x2.

Notebook dùng đúng dev-200 của M48, checkpoint từng câu và chỉ báo promotion khi METEOR tăng. Nó không chạy public và không build lại DB.

In [ ]:
from hashlib import sha256
from pathlib import Path
import runpy
import sys
from zipfile import ZipFile

input_root = Path('/kaggle/input')
working = Path('/kaggle/working')
scripts = sorted(input_root.rglob('m49_kaggle_candidate_dev.py'))
if not scripts:
    zips = sorted(input_root.rglob('legal-agentic-rag-m49-source.zip'))
    assert len(zips) == 1, f'Cần đúng 1 source ZIP M49, tìm thấy: {zips}'
    bootstrap = working / 'm49-dev-bootstrap'
    if not bootstrap.is_dir():
        with ZipFile(zips[0]) as archive:
            archive.extractall(bootstrap)
    scripts = sorted(bootstrap.rglob('m49_kaggle_candidate_dev.py'))
assert scripts, 'Không tìm thấy dev runner M49'
digests = {sha256(path.read_bytes()).hexdigest() for path in scripts}
assert len(digests) == 1, f'Có nhiều source M49 khác nhau: {scripts}'
script = scripts[0]
sys.path.insert(0, str(script.parent))
runpy.run_path(str(script), run_name='__main__')
